# Bài toán 1: Dự đoán kết quả trận đấu (Phân tách Premium vs Basic Model)

Sau khi EDA, ta nhận thấy dữ liệu OpenLiga bị thiếu Odds và xG. Ta sẽ chia ra huấn luyện 2 mô hình khác nhau để tối ưu hóa đặc trưng thay vì nhét chung tất cả vào 1 mô hình XGBoost (dù XGBoost xử lý được NaN nhưng sẽ làm loãng trọng số của Odds).

In [ ]:
!pip install -q mlflow==3.16.1 pysocks kaggle xgboost shap duckdb

In [ ]:
# Upload file colab_utils.py từ máy tính
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

import importlib
import colab_utils
importlib.invalidate_caches()
importlib.reload(colab_utils)
print("File đang dùng:", colab_utils.__file__)


In [ ]:
import duckdb
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score
import colab_utils
from google.colab import userdata

colab_utils.install_tailscale()
colab_utils.setup_tailscale(auth_key=userdata.get('TAILSCALE_AUTH_KEY'))
colab_utils.setup_mlflow("http://20.41.113.183:8080", experiment="Match-Prediction-DualModels")

con = duckdb.connect(':memory:')
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET minio_secret (
        TYPE S3, KEY_ID '{userdata.get('MINIO_ACCESS_KEY')}', SECRET '{userdata.get('MINIO_SECRET_KEY')}',
        ENDPOINT '20.41.113.183:9000', URL_STYLE 'path', USE_SSL 'false'
    );
""")

# Đọc kết hợp 2 bảng Match Features
df = con.execute("""
    SELECT m.*, t.f_form5 
    FROM read_parquet('s3://football-lake/dwh/mart_ml_match_features.parquet') m
    LEFT JOIN read_parquet('s3://football-lake/dwh/mart_ml_team_match_features.parquet') t
    ON m.match_key = t.match_key AND t.side = 'home'
    WHERE m.is_warm = true AND m.is_labeled = true
""").df()
df['target'] = df['y_result_code'].astype(int)


In [ ]:
# KHẮC PHỤC DỮ LIỆU: Tách 2 tập (Premium vs Basic)
# Premium: 6 giải lớn có xác suất thị trường (Odds)
# Basic: Bao gồm OpenLiga (Không dùng Odds)

df_premium = df[df['f_mkt_p_home'].notnull()].copy()
df_basic = df.copy() # Dùng toàn bộ data

# Features
feat_premium = [c for c in df_premium.columns if c.startswith('f_')]
# Bỏ các cột Odds và Wiki, Age khỏi Basic vì OpenLiga ko có
feat_basic = [c for c in df_basic.columns if c.startswith('f_') and not c.startswith('f_mkt_') and not c.startswith('f_wiki_')]

print(f"Premium data: {len(df_premium)} trận | Basic data: {len(df_basic)} trận")


In [ ]:
import mlflow
from sklearn.metrics import classification_report, log_loss

def train_eval(data, features, model_name):
    print(f"\n--- Training {model_name} ---")
    train = data[data['split'] == 'train']
    valid = data[data['split'] == 'valid']
    test = data[data['split'] == 'test']
    
    # Định nghĩa tham số mô hình
    params = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': 300,
        'random_state': 42
    }
    
    with mlflow.start_run(run_name=model_name):
        # 1. Log tham số (Parameters)
        mlflow.log_params(params)
        mlflow.log_param("n_features", len(features))
        
        # 2. Khởi tạo & Huấn luyện
        model = xgb.XGBClassifier(**params)
        model.fit(train[features], train['target'], 
                  eval_set=[(valid[features], valid['target'])], 
                  verbose=False)
        
        # 3. Đánh giá (Metrics)
        # Trên Test
        preds = model.predict(test[features])
        probs = model.predict_proba(test[features])
        acc_test = accuracy_score(test['target'], preds)
        loss_test = log_loss(test['target'], probs)
        
        # Trên Valid
        acc_valid = accuracy_score(valid['target'], model.predict(valid[features]))
        
        # Log Metrics
        mlflow.log_metrics({
            "test_accuracy": acc_test,
            "test_log_loss": loss_test,
            "valid_accuracy": acc_valid
        })
        
        print(f"[{model_name}] Test Acc: {acc_test:.4f} | Valid Acc: {acc_valid:.4f}")
        print("\nClassification Report (Test):\n", classification_report(test['target'], preds))
        
        # 4. Lưu Model vào MLflow
        model_info = mlflow.xgboost.log_model(model, artifact_path="model")
        print(f"Đã lưu mô hình tại URI: {model_info.model_uri}")
        
        # Lưu các giá trị mẫu để tí nữa test tải model
        return model_info.model_uri, test

# Thực thi Train cho cả 2 nhánh (Premium vs Basic)
uri_premium, test_premium = train_eval(df_premium, feat_premium, "Premium_Model_With_Odds")
uri_basic, test_basic = train_eval(df_basic, feat_basic, "Basic_Model_Form_Only")


In [ ]:
# ==========================================
# 5. TẢI MÔ HÌNH VỀ TỪ MLFLOW ĐỂ DÙNG THỬ (INFERENCE)
# ==========================================
import mlflow.xgboost

print("Đang tải lại mô hình Premium từ MLflow...")
loaded_model = mlflow.xgboost.load_model(uri_premium)

# Thử nghiệm dự đoán trên 5 trận đầu tiên của tập Test
sample_data = test_premium.head(5)
sample_features = sample_data[feat_premium]

print("Dự đoán 5 trận đầu (0=Sân khách, 1=Hòa, 2=Sân nhà):")
predictions = loaded_model.predict(sample_features)
probabilities = loaded_model.predict_proba(sample_features)

for i in range(5):
    print(f"Trận {i+1}:")
    print(f" - Nhãn thực tế: {sample_data['target'].iloc[i]}")
    print(f" - Mô hình dự đoán: {predictions[i]}")
    print(f" - Xác suất [Khách, Hòa, Nhà]: {probabilities[i]}\n")
